In [19]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.linear_model import SGDClassifier

In [20]:
df = pd.read_csv('../data/Resume_cleaned.csv')

print("Original Shape:", df.shape)
df.head()

Original Shape: (2484, 6)


,ID,Resume_str,Resume_html,Category,cleaned,word_count
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR,administrator marketing associate administrato...,495
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR,specialist operations summary versatile media ...,523
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR,director summary years experience recruiting p...,701
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR,specialist summary dedicated driven dynamic ye...,259
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR,manager skill highlights skills department sta...,845


In [21]:
# Remove null values
df = df.dropna(subset=['cleaned'])

# Convert to string
df['cleaned'] = df['cleaned'].astype(str)

# Keep categories having at least 50 resumes
counts = df['Category'].value_counts()

valid_classes = counts[counts >= 50].index

df = df[df['Category'].isin(valid_classes)]

print("Filtered Shape:", df.shape)
print(df['Category'].value_counts())

Filtered Shape: (2425, 6)
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      119
ADVOCATE                  118
ACCOUNTANT                118
FINANCE                   118
CHEF                      118
ENGINEERING               118
FITNESS                   117
AVIATION                  117
SALES                     116
HEALTHCARE                115
BANKING                   115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
Name: count, dtype: int64


In [22]:
X = df['cleaned']
y = df['Category']

In [23]:
vectorizer = TfidfVectorizer(
    max_features=12000,
    stop_words='english',
    ngram_range=(1,2),
    min_df=2,
    sublinear_tf=True
)

X_vec = vectorizer.fit_transform(X)

print("Vector Shape:", X_vec.shape)

Vector Shape: (2425, 12000)


In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vec,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)

(1940, 12000) (485, 12000)


In [25]:
model = SGDClassifier(
    loss='hinge',
    alpha=1e-4,
    max_iter=2000,
    random_state=42
)

model.fit(X_train, y_train)

print("Model Trained Successfully")

Model Trained Successfully


In [26]:
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)

print("Accuracy:", acc)

Accuracy: 0.7010309278350515


In [27]:
print(classification_report(y_test, pred, zero_division=0))

                        precision    recall  f1-score   support

            ACCOUNTANT       0.81      0.88      0.84        24
              ADVOCATE       0.75      0.75      0.75        24
           AGRICULTURE       0.75      0.46      0.57        13
               APPAREL       0.54      0.37      0.44        19
                  ARTS       0.56      0.43      0.49        21
              AVIATION       0.70      0.70      0.70        23
               BANKING       0.62      0.78      0.69        23
  BUSINESS-DEVELOPMENT       0.58      0.79      0.67        24
                  CHEF       0.94      0.71      0.81        24
          CONSTRUCTION       0.87      0.91      0.89        22
            CONSULTANT       0.74      0.61      0.67        23
              DESIGNER       0.89      0.76      0.82        21
         DIGITAL-MEDIA       0.67      0.53      0.59        19
           ENGINEERING       0.66      0.79      0.72        24
               FINANCE       0.95      

In [28]:
joblib.dump(model, '../src/model.pkl')
joblib.dump(vectorizer, '../src/vectorizer.pkl')

print("Saved model.pkl and vectorizer.pkl")

Saved model.pkl and vectorizer.pkl
